# ПЗ-4. Суммаризация новостей и качество данных
В этой практической работе рассмотрим, как изменение источника новостей и ошибки подготовки данных влияют на работу готовой языковой модели. На примере последовательно разберем:
- оценку качества резюме на исходной выборке новостей с помощью ROUGE;
- влияние смены источника данных на результаты модели;
- адаптацию prompt с помощью примеров из разных источников;
- моделирование ошибки сборки, при которой в новость попадают посторонние абзацы;
- фильтрацию посторонних абзацев и оценку качества фильтра;
- сравнение резюме по чистым, загрязненным и отфильтрованным текстам;
- сохранение версий данных и prompt в MLflow.
Главная цель практики: понять, почему качество модели может меняться при переходе к новым данным, как ошибки подготовки текста влияют на резюме и почему пользу фильтрации нужно проверять не только по метрикам классификации, но и по качеству конечного результата.

## Зависимости и импорты


In [ ]:
%pip install evaluate rouge_score

## Подготовка

Ноутбук использует локальную GGUF-модель через `llama-cpp-python`. Датасеты загружаются из Hugging Face, эксперименты сохраняются в локальном MLflow. В демонстрационном запуске используются 100 новостей.
Ячейка установки ниже рассчитана **только на Google Colab**. Она ставит сборку `llama-cpp-python` с поддержкой CUDA 12.4 и подгружает библиотеки CUDA по путям, которые существуют только в окружении Colab (`/usr/local/lib/python*/dist-packages`). Поэтому при локальном запуске ее выполнять не нужно, а подготовку окружения следует сделать вручную.
Для NVIDIA вместо сборки можно взять готовую сборку: `pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124`.
Проверьте, что ускорение на GPU доступно:

```python
from llama_cpp import llama_supports_gpu_offload
print(llama_supports_gpu_offload())
```

Если результат `False`, а видеокарта есть, библиотеки CUDA не найдены: проверьте установку CUDA и переменную `LD_LIBRARY_PATH`.
Если GPU нет, в ячейках подключения моделей замените `n_gpu_layers=-1` на `n_gpu_layers=0` и уменьшите `N_EVAL`: на CPU генерация заметно медленнее.



In [ ]:
%pip install -q datasets huggingface_hub mlflow nltk rouge-score pandas matplotlib tqdm

%pip uninstall -y llama-cpp-python

%pip install -q --no-cache-dir \
  nvidia-cuda-runtime-cu12 \
  nvidia-cublas-cu12

%pip install -q --no-cache-dir \
  llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

import glob
import ctypes

names = [
    "libcudart.so.12",
    "libcublas.so.12",
    "libcublasLt.so.12",
]

for name in names:
    files = glob.glob(
        f"/usr/local/lib/python*/dist-packages/nvidia/**/{name}",
        recursive=True,
    )
    if files:
        ctypes.CDLL(files[0], mode=ctypes.RTLD_GLOBAL)

from llama_cpp import llama_supports_gpu_offload

print("GPU offload support:", llama_supports_gpu_offload())

In [ ]:
import sys
from tqdm import tqdm

import json, random, re
import numpy as np
import pandas as pd
import mlflow
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
# Для полной версии практики используйте 500-1000 новостей из каждого корпуса.
CONFIG = dict(N_NEWS=100, SEED=42, P_NOISE=1.0, N_NOISE=3,
              DONOR_SAME_CORPUS_SHARE=0.5, MAX_PARAGRAPHS=8,
              EVAL_N=10, TRAIN_SHARE=0.8, VALIDATION_SHARE=0.1, TEST_SHARE=0.1)
random.seed(CONFIG['SEED']);
np.random.seed(CONFIG['SEED']);
assert CONFIG['N_NEWS'] >= 3 and CONFIG['MAX_PARAGRAPHS'] > CONFIG['N_NOISE'] >= 1
assert 0 <= CONFIG['P_NOISE'] <= 1 and 0 <= CONFIG['DONOR_SAME_CORPUS_SHARE'] <= 1
assert abs(sum(CONFIG[k] for k in ('TRAIN_SHARE', 'VALIDATION_SHARE', 'TEST_SHARE')) - 1) < 1e-9
assert CONFIG['EVAL_N'] >= 1

In [ ]:
from llama_cpp import Llama

SEED = 42
MODEL_REPO = "Qwen/Qwen3-8B-GGUF"
MODEL_FILENAME = "Qwen3-8B-Q4_K_M.gguf"
TEXT_CHAR_LIMIT = 8000
EXAMPLE_CHAR_LIMIT = 700
MAX_TOKENS = 256
LLM = Llama.from_pretrained(
    repo_id=MODEL_REPO,
    filename=MODEL_FILENAME,
    n_ctx=12000,
    n_gpu_layers=-1,
    verbose=False,
)


def ask_llm(prompt: str, temperature: float = 0.0, max_tokens: int = MAX_TOKENS, llm=LLM) -> str:
    """Отправляет prompt локальной модели.

    Args:
        prompt: Инструкция и исходные данные.
        temperature: Степень случайности генерации.
        max_tokens: Максимальная длина ответа в токенах.

    Returns:
        Текст ответа модели.
    """
    response = llm.create_chat_completion(
        messages=[{"role": "user", "content": f"{prompt}\n/no_think"}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    answer = response["choices"][0]["message"]["content"] or ""
    answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.DOTALL)
    answer = re.sub(r"<think>.*", "", answer, flags=re.DOTALL)
    return answer.strip()


## 2. Часть 1. Эталонная суммаризация на Gazeta


Как и в предыдущих практических занятиях, мы работаем с новостным датасетом и решаем задачу суммаризации: по полному тексту новости необходимо составить краткое резюме. В первой части используется корпус Gazeta, в котором каждой новости соответствует эталонное резюме, составленное человеком.

Загрузим тексты новостей вместе с эталонными резюме и разделим их на train, validation и test. Разбиение фиксируется и используется во всех последующих экспериментах, что позволяет сравнивать результаты на одних и тех же данных.

In [ ]:
def collect(rows, corpus):
    result = []
    for row in rows:
        text = (row.get('text') or '').strip()
        reference = (row.get('summary') or '').strip()
        if text and reference:
            result.append({'id': f'{corpus}:{len(result)}', 'title': row.get('title') or '',
                           'text': text, 'reference': reference, 'corpus': corpus})
        if len(result) == CONFIG['N_NEWS']:
            break
    assert len(result) == CONFIG['N_NEWS'], f'{corpus}: недостаточно новостей с резюме'
    return result


gazeta = collect(load_dataset('IlyaGusev/gazeta', split='train', streaming=True), 'Gazeta')
print('Загружено новостей:', len(gazeta))

### Разбиение по новостям

Зафиксируем части train, validation и test. В этом блоке используется только test; остальные части используются для примеров и подготовки данных.

In [ ]:
def split_news(rows):
    rows = list(rows)
    random.shuffle(rows)
    total = len(rows)
    nv = max(1, round(total * CONFIG['VALIDATION_SHARE']))
    nt = max(1, round(total * CONFIG['TEST_SHARE']))
    return {'train': rows[:total - nv - nt], 'validation': rows[total - nv - nt:total - nt], 'test': rows[total - nt:]}


gazeta_splits = split_news(gazeta)
print({s: len(rows) for s, rows in gazeta_splits.items()})

### Промпт и версия данных

Результат суммаризации определяется не только моделью, но и двумя компонентами: промптом и данными, на которых он проверяется. Изменение формулировки инструкции, набора примеров или состава тестовой выборки меняет метрики так же, как смена модели. Поэтому и промпты, и датасеты необходимо версионировать, как версионируется код.

Версионирование нужно по нескольким причинам:

- **Воспроизводимость.** Для любого полученного значения метрики можно точно указать, какой промпт и какая версия данных его дали.
- **Корректное сравнение.** Различия между экспериментами можно относить к конкретному изменению, если все остальное зафиксировано.
- **Возврат к прежнему состоянию.** Если новая версия промпта или данных ухудшила результат, можно вернуться к предыдущей.

В практике используется MLflow. Промпт регистрируется в Prompt Registry: каждое изменение создает новую версию с адресом вида `prompts:/имя/номер`, по которому его можно загрузить. Датасет сохраняется как Evaluation Dataset с отдельным идентификатором, а полный снимок дополнительно записывается в артефакты запуска. Каждый запуск создает новый датасет, поэтому снимки разных запусков не смешиваются.


In [ ]:
few_shot_examples = gazeta_splits['train'][:2]
few_shot_ids = [row['id'] for row in few_shot_examples]
assert not set(few_shot_ids) & {row['id'] for s in ('validation', 'test') for row in gazeta_splits[s]}
SUMMARY_PROMPT = ('Составь краткое фактическое резюме новости на русском языке. '
                  'Используй только сведения из текста. Верни только резюме.')
for i, row in enumerate(few_shot_examples, start=1):
    SUMMARY_PROMPT += (f"\n\nПример {i}.\n"
                       f"\nТекст: {row['text']}\nРезюме: {row['reference']}")
print('ID примеров промпта:', few_shot_ids)

mlflow.set_tracking_uri('sqlite:///mlflow_pz4.db')
mlflow.set_experiment('pz4_news_summarization')

# Регистрация создает новую версию промпта; ее URI сохраняется со снимком.
registered_summary = mlflow.genai.register_prompt(
    name='pz4_news_summarization', template=SUMMARY_PROMPT,
    commit_message='Few-shot примеры из train Gazeta')
summary_prompt_uri = f'prompts:/{registered_summary.name}/{registered_summary.version}'
print('Промпт:', summary_prompt_uri)

In [ ]:
from mlflow.genai.datasets import create_dataset
from uuid import uuid4


def snapshot_dataset(name, splits, prompts, prompt_uri=None):
    """Сохраняет версию данных в MLflow как Evaluation Dataset и снимок запуска.
    Создает новый Evaluation Dataset с уникальным именем (к имени версии
    добавляется случайный суффикс).

    Args:
        name: Название версии данных, например 'v1_gazeta_clean'.
        splits: Словарь {часть: список новостей} с ключами train, validation, test.
        prompts: Словарь с текстами промптов и идентификаторами примеров;
            сохраняется в артефакты запуска.

    Returns:
        Идентификатор созданного Evaluation Dataset.
    """
    prompt_uri = prompt_uri or summary_prompt_uri
    experiment_id = mlflow.get_experiment_by_name('pz4_news_summarization').experiment_id
    dataset = create_dataset(name=f'{name}_{uuid4().hex[:12]}', experiment_id=[experiment_id],
                             tags={'dataset_version': name, 'summary_prompt_uri': prompt_uri})
    records = []
    for split, rows in splits.items():
        for row in rows:
            inputs = {'split': split, **{k: v for k, v in row.items() if k != 'reference'}}
            records.append({'inputs': inputs, 'expectations': {'expected_response': row.get('reference', '')}})
    dataset.merge_records(records)
    with mlflow.start_run(run_name=name) as run:
        mlflow.log_params({'dataset_version': name, 'dataset_id': dataset.dataset_id,
                           'summary_prompt_uri': prompt_uri, **CONFIG})
        mlflow.log_dict(splits, 'dataset.json')
        mlflow.log_dict(prompts, 'prompts.json')
        print('Evaluation Dataset:', dataset.name, dataset.dataset_id, 'версия:', dataset.version)
        print('Снимок:', f'runs:/{run.info.run_id}/dataset.json')
        return dataset.dataset_id


v1_dataset_id = snapshot_dataset('v1_gazeta_clean', gazeta_splits,
                                 {'summary': SUMMARY_PROMPT, 'few_shot_ids': few_shot_ids})
print('ID датасета:', v1_dataset_id)

### Модель и эталонная оценка

Оценим качество суммаризации на тестовой выборке Gazeta: для каждой новости построим резюме и сравним его с эталоном по ROUGE-1 и ROUGE-2. Полученные значения служат эталонной оценкой, или baseline. С ними будут сравниваться все последующие эксперименты.


In [ ]:
import evaluate
from mlflow.genai.datasets import get_dataset

SUMMARY_PROMPT = mlflow.genai.load_prompt(summary_prompt_uri).template
v1_records = get_dataset(dataset_id=v1_dataset_id).to_df()
v1 = {'train': [], 'validation': [], 'test': []}
for record in v1_records.to_dict('records'):
    inputs = dict(record['inputs'])
    split = inputs.pop('split')
    inputs['reference'] = record['expectations']['expected_response']
    v1[split].append(inputs)

# Библиотечная реализация ROUGE; токенизация сохраняет русские слова.
rouge = evaluate.load('rouge')


def summarize(item):
    text = '\n'.join(item['paragraphs']) if 'paragraphs' in item else item['text']
    return ask_llm(f"{SUMMARY_PROMPT}\nТекст: {text}\nРезюме:")


def summary_results(items):
    return [{'id': item['id'], 'corpus': item['corpus'], 'reference': item['reference'],
             'summary': summarize(item)} for item in items]


def summary_metrics(rows):
    scores = rouge.compute(predictions=[row['summary'] for row in rows],
                           references=[row['reference'] for row in rows],
                           rouge_types=['rouge1', 'rouge2'],
                           tokenizer=lambda text: re.findall(r'\w+', text.lower(), flags=re.UNICODE))
    return {'rouge1_f1': scores['rouge1'], 'rouge2_f1': scores['rouge2'],
            'mean_chars': np.mean([len(row['summary']) for row in rows])}


def log_metrics(name, values):
    with mlflow.start_run(run_name=name):
        mlflow.log_metrics({key: float(value) for key, value in values.items()})


In [ ]:

test_gazeta = v1['test'][:CONFIG['EVAL_N']]
baseline_gazeta = summary_results(test_gazeta)
print(pd.DataFrame(baseline_gazeta))
print('Эталон Gazeta:', summary_metrics(baseline_gazeta))
log_metrics('v1_summary', summary_metrics(baseline_gazeta))

## 3. Дрейф источника данных (covariate shift)

Качество модели оценивается на данных, похожих на те, для которых настроены промпт и правила подготовки. В эксплуатации распределение данных со временем меняется, и значения метрик, полученные на исходной выборке, перестают описывать реальное качество системы. Такое изменение называют **дрейфом данных** (dataset shift). Пусть $x$ - вход модели, $y$ - правильный ответ; для суммаризации $x$ - текст новости, $y$ - эталонное резюме. Различают три основных вида дрейфа.

- **Covariate shift** - изменяется распределение входных данных $P(x)$, а связь между входом и ответом $P(y \mid x)$ остается прежней. Например, в систему начинают поступать новости другого издания, другой тематики или другой длины.
- **Prior shift** - изменяется распределение ответов $P(y)$, например доля классов в задаче классификации.
- **Concept drift** - изменяется сама связь между входом и ответом $P(y \mid x)$. Например, меняются требования к резюме: оно должно быть короче или иметь другую структуру.

В этом разделе моделируется **covariate shift**: модель, промпт и правила подготовки данных остаются прежними, а вместо Gazeta на вход поступают новости из корпуса XL-Sum. На практике такая ситуация возникает при подключении нового источника или поставщика новостей, смене сайта или формата выгрузки, изменении тематики или периода публикаций, а также при переходе к другому стилю изложения.

Важно учитывать ограничение эксперимента. В реальных системах сдвиг в чистом виде встречается редко, и здесь корпуса различаются не только текстами новостей, но и эталонными резюме: в XL-Sum они, как правило, короче, чем в Gazeta. Поэтому разница в метриках отражает одновременно сдвиг входных данных и различие в способе составления эталонов.

В качестве точки отсчета используется версия `v1_gazeta_clean`. Применим к XL-Sum тот же промпт и ту же процедуру оценки, затем сравним метрики с результатами на Gazeta.

In [ ]:
xlsum_url = ('https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/'
             '8722367c537146b056c6550160930d9e12e75ce7/russian/test-00000-of-00001.parquet')
xlsum = collect(load_dataset('parquet', data_files=xlsum_url, split='train', streaming=True), 'XL-Sum')
xlsum_splits = split_news(xlsum)
corpus_splits = {'Gazeta': gazeta_splits, 'XL-Sum': xlsum_splits}
v2 = {s: gazeta_splits[s] + xlsum_splits[s] for s in ('train', 'validation', 'test')}
print({s: len(rows) for s, rows in v2.items()})
v2_dataset_id = snapshot_dataset('v2_xlsum_gazeta_clean', v2, {'summary': SUMMARY_PROMPT, 'few_shot_ids': few_shot_ids})

In [ ]:
v2_records = get_dataset(dataset_id=v2_dataset_id).to_df()
test_xlsum = []
for record in v2_records.to_dict('records'):
    inputs = dict(record['inputs'])
    if inputs.pop('split') == 'test' and inputs['corpus'] == 'XL-Sum':
        inputs['reference'] = record['expectations']['expected_response']
        test_xlsum.append(inputs)
test_xlsum = test_xlsum[:CONFIG['EVAL_N']]
baseline_xlsum = summary_results(test_xlsum)
source_comparison = pd.DataFrame({'Gazeta (эталон v1)': summary_metrics(baseline_gazeta),
                                  'XL-Sum (v2)': summary_metrics(baseline_xlsum)}).T
print(source_comparison)
log_metrics('v2_xlsum_summary', summary_metrics(baseline_xlsum))

Сравните метрики двух источников в таблице выше. Разница отражает не только тексты новостей, но и различия в эталонных резюме; на небольшой выборке ее нельзя надежно обобщить на весь корпус.


**Задание 1.** Адаптируйте промпт к двум источникам.

1. Соберите новую версию датасета, содержащую новости обоих корпусов, Gazeta и XL-Sum, и зарегистрируйте ее в MLflow как Evaluation Dataset.
2. Составьте новый промпт с примерами из обоих корпусов. Примеры берите только из train. Зарегистрируйте промпт в MLflow Prompt Registry.
3. Оцените оба промпта, исходный и новый, на тестовых новостях каждого корпуса.
4. Постройте столбчатые диаграммы ROUGE-1 и ROUGE-2: по горизонтали корпуса, столбцы соответствуют промптам.
5. Объясните, как примеры из второго корпуса повлияли на результат на Gazeta и на XL-Sum.


## 4. Часть 2. Ошибка сборки: посторонние абзацы

В части 1 данные были корректными: менялся только их источник. Другая проблема в эксплуатации возникает в самом пайплайне подготовки данных. Из-за ошибки на этапе сбора или обработки входной текст оказывается искажен. Это дефект качества данных, а не естественное изменение их распределения, и исправлять его нужно в пайплане, а не адаптацией модели.

Типичные причины таких ошибок:

- сбой парсинга веб-страницы: в текст новости попадают блоки «Читайте также», фрагменты соседних материалов, комментарии и реклама;
- неверное объединение таблиц, например по неправильному ключу, когда новости получают чужие фрагменты;
- сдвиг строк или дублирование записей при выгрузке и конвертации;
- ошибки сегментации и склейки длинных текстов по абзацам.

Главная опасность в том, что такая ошибка не вызывает сбоя: модель получает текст нормального вида и честно по нему суммирует. Посторонние сведения могут попасть в резюме, а привычные метрики качества при этом изменятся слабо.

Смоделируем такую ситуацию: в каждую новость добавим абзацы из других новостей. Модель, промпт и эталонное резюме остаются прежними, поэтому все изменения качества вызваны только загрязнением входа. Затем рассмотрим пример испорченной новости.

### Подготовка абзацев

Разделим тексты новостей на условные абзацы.

In [ ]:
def split_paragraphs(text, target_chars=500):
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]
    paragraphs, current = [], []
    for sentence in sentences:
        current.append(sentence)
        if len(' '.join(current)) >= target_chars:
            paragraphs.append(' '.join(current))
            current = []
    if current:
        paragraphs.append(' '.join(current))
    limit = CONFIG['MAX_PARAGRAPHS'] - CONFIG['N_NOISE']
    while len(paragraphs) > limit:
        i = min(range(len(paragraphs) - 1), key=lambda j: len(paragraphs[j]) + len(paragraphs[j + 1]))
        paragraphs[i:i + 2] = [paragraphs[i] + ' ' + paragraphs[i + 1]]
    return paragraphs


segmented = {s: [{**row, 'paragraphs': split_paragraphs(row['text'])} for row in rows]
             for s, rows in v2.items()}
assert all(x['paragraphs'] for rows in segmented.values() for x in rows)
print('Пример сегментации:', segmented['train'][0]['paragraphs'][:2])

### Имитируем ошибку

Добавим в каждую новость посторонние абзацы из других статей: по `N_NOISE` абзацев в случайные позиции текста. Доля загрязненных новостей задается параметром `P_NOISE`.

Доноры выбираются по следующим правилам:

- донором служит другая новость из того же раздела: train, validation или test. Абзацы не пересекают границы разделов, поэтому тестовые новости не получают информации из обучающих и наоборот;
- новость не может быть донором самой себе;
- половина вставок берется из новостей того же корпуса, половина из другого (параметр `DONOR_SAME_CORPUS_SHARE`). Абзацы из своего корпуса сложнее отличить: у них похожи стиль и тематика. Если подходящего донора нужного корпуса в разделе нет, берется любой другой.

Каждому абзацу присваивается метка: исходные абзацы получают `relevant`, вставленные `irrelevant`. Метки сохраняют происхождение абзаца и нужны для проверки эксперимента и оценки фильтра. Модель их не получает.

In [ ]:
def contaminate(row, pool, p_noise=CONFIG['P_NOISE']):
    item = {**row,
            'paragraphs': [{'text': p, 'label': 'relevant', 'source_id': row['id'], 'corpus': row['corpus']} for p in
                           row['paragraphs']]}
    if random.random() < p_noise:
        for _ in range(CONFIG['N_NOISE']):
            same = random.random() < CONFIG['DONOR_SAME_CORPUS_SHARE']
            candidates = [x for x in pool if x['id'] != row['id'] and (x['corpus'] == row['corpus']) == same]
            if not candidates: candidates = [x for x in pool if x['id'] != row['id']]
            assert candidates, 'В split нет допустимого донора'
            donor = random.choice(candidates)
            position = random.randrange(len(item['paragraphs']) + 1)
            item['paragraphs'].insert(position, {'text': random.choice(donor['paragraphs']), 'label': 'irrelevant',
                                                 'source_id': donor['id'], 'corpus': donor['corpus']})
    item['text'] = '\n'.join(p['text'] for p in item['paragraphs'])
    return item


v3 = {s: [contaminate(x, segmented[s]) for x in segmented[s]] for s in segmented}

# Фиксируем загрязненные данные до их загрузки в следующих ячейках.
v3_dataset_id = snapshot_dataset('v3_contaminated', v3,
                                 {'summary': SUMMARY_PROMPT, 'few_shot_ids': few_shot_ids})

### Как выглядит ошибка?

Найдем первую загрязненную новость и выведем ее заголовок и абзацы вместе с метками. Метки показывают происхождение каждого абзаца: `relevant` означает исходный абзац новости, `irrelevant` означает вставку из другой статьи. Они нужны для проверки эксперимента, модель их не получает.

Обратите внимание, как выглядят вставки. Это связные тексты без технических признаков ошибки: в них нет обрывов, служебных символов и разметки. Часто такой абзац можно принять за часть новости, особенно если донор из того же корпуса и тема похожа. Именно поэтому подобные ошибки трудно заметить без специальных проверок. Попробуйте сначала определить посторонние абзацы самостоятельно по заголовку, а затем сверьтесь с метками.

In [ ]:
example = next(
    (news for rows in v3.values() for news in rows if any(p['label'] == 'irrelevant' for p in news['paragraphs'])),
    None)
if example is None:
    print('В этой выборке загрязненных новостей не оказалось.')
else:
    print('Заголовок:', example['title'])
    for i, paragraph in enumerate(example['paragraphs'], 1):
        print(f"\nАбзац {i} - {paragraph['label']}\n{paragraph['text']}")

### Как ошибка влияет на суммаризацию?

Сравним резюме до и после добавления посторонних абзацев. Для каждой загрязненной новости построим чистый вариант, убрав вставки; порядок исходных абзацев одинаков в обоих вариантах. Модель, промпт и эталонное резюме не меняются. Оцениваем только test, не более `EVAL_N` новостей.

In [ ]:
v3_records = get_dataset(dataset_id=v3_dataset_id).to_df().to_dict('records')
# Сортировка по id делает выборку одинаковой при каждом запуске.
noise_test = sorted([r for r in v3_records if r['inputs']['split'] == 'test'
                     and any(p['label'] == 'irrelevant' for p in r['inputs']['paragraphs'])],
                    key=lambda r: r['inputs']['id'])[:CONFIG['EVAL_N']]
assert noise_test, 'В test нет загрязненных новостей. Пересоздайте версии данных.'
prompt = mlflow.genai.load_prompt(summary_prompt_uri).template


def clean_and_noisy_text(news):
    """Возвращает текст новости без вставок и с вставками."""
    clean = '\n'.join(p['text'] for p in news['paragraphs'] if p['label'] == 'relevant')
    noisy = '\n'.join(p['text'] for p in news['paragraphs'])
    return clean, noisy


# Пример на одной новости.
row = noise_test[min(4, len(noise_test) - 1)]
news = row['inputs']
clean_text, noise_text = clean_and_noisy_text(news)
print(noise_text)
print('Заголовок:', news['title'])
print('\nВставленные абзацы:')
for p in news['paragraphs']:
    if p['label'] == 'irrelevant':
        print('-', p['text'][:300])
print('\nЭталон:', row['expectations']['expected_response'])
print('\nРезюме по чистой новости:', ask_llm(f"{prompt}\n\nТекст: {clean_text}\nРезюме:"))
print('\nРезюме по новости с мусором:', ask_llm(f"{prompt}\n\nТекст: {noise_text}\nРезюме:"))

Сравните два резюме и вставленные абзацы в выводе выше. Проверьте, появились ли в резюме сведения из вставок и сохранились ли факты исходной новости. Результат зависит от выбранной новости и текущего запуска модели.


**Задание 2.** Оцените влияние посторонних абзацев на качество суммаризации.

1. Постройте резюме для всех загрязненных новостей из `noise_test` в двух вариантах: по чистому тексту и по тексту с вставками. Модель и промпт одинаковые.
2. Вычислите ROUGE-1, ROUGE-2 и среднюю длину резюме для каждого варианта и сравните их с показателями на чистых новостях. На сколько изменились метрики?
3. Постройте столбчатую диаграмму: по горизонтали метрики, столбцы соответствуют чистым новостям и новостям с мусором.


### Фильтрация посторонних абзацев

Посторонние абзацы искажают резюме, поэтому их полезно обнаружить до суммаризации. Это задача классификации: для каждого абзаца необходимо определить, относится ли он к новости. Решим ее той же языковой моделью, без дообучения, с помощью промпта.

Модели передаются заголовок и пронумерованные абзацы новости. Она возвращает метку `relevant` или `irrelevant` для каждого абзаца в формате JSON. Абзацы с меткой `irrelevant` удаляются, остальные передаются на суммаризацию. Определять посторонние абзацы только по заголовку и содержанию соседних абзацев непросто: вставка из новости на близкую тему может выглядеть уместной, а короткий заголовок дает мало информации.

Строгий формат ответа нужен для автоматической обработки, но маленькая модель не всегда его соблюдает. Ответ считается некорректным, если он не разбирается как JSON, число меток не совпадает с числом абзацев или встречаются недопустимые значения. В этом случае фильтр ничего не удаляет и сохраняет все абзацы: ошибочное удаление нужного текста вреднее, чем пропуск постороннего. Такие ответы учитываются отдельно как доля некорректных ответов (`invalid_json_rate`), поскольку это самостоятельный показатель надежности фильтра.

In [ ]:
FILTER_PROMPT = (
    'Прочитай абзацы новости. В текст могут быть случайно вставлены посторонние абзацы из других новостей. '
    'Для каждого абзаца определи, относится ли он к заголовку. '
    'Верни только JSON: {"labels":["relevant" или "irrelevant", ...]}.')

In [ ]:
def filter_messages(item, prompt=FILTER_PROMPT):
    numbered = '\n'.join(f'{i + 1}. {p["text"]}' for i, p in enumerate(item['paragraphs']))
    return [{'role': 'user', 'content': f"{prompt}\n\nАбзацы:\n{numbered}"}]


def parse_labels(text, size):
    text = text.replace('```json', '').replace('```', '')
    try:
        labels = json.loads(text.strip())['labels']
        if not isinstance(labels, list) or len(labels) != size or any(
                x not in ('relevant', 'irrelevant') for x in labels):
            raise ValueError('Некорректные метки')
        return labels, False
    except (json.JSONDecodeError, KeyError, TypeError, ValueError):
        print(text)
        return ['relevant'] * size, True


def filter_news(item, generate=ask_llm, prompt=FILTER_PROMPT):
    answer = generate(filter_messages(item, prompt)[0]['content'])
    labels, invalid = parse_labels(answer, len(item['paragraphs']))
    kept = [p['text'] for p, label in zip(item['paragraphs'], labels) if label == 'relevant']
    return {'true': [p['label'] for p in item['paragraphs']], 'pred': labels, 'invalid': invalid,
            'cleaned': {**item, 'paragraphs': kept}}


def filter_metrics(rows):
    true = [label for row in rows for label in row['true']]
    pred = [label for row in rows for label in row['pred']]
    precision, recall, f1, _ = precision_recall_fscore_support(true, pred, average='binary', pos_label='irrelevant',
                                                               zero_division=0)
    return {'accuracy': accuracy_score(true, pred), 'precision': precision, 'recall': recall, 'f1': f1,
            'invalid_json_rate': np.mean([row['invalid'] for row in rows])}

### Оценка фильтра

Оценим фильтр на тех же загрязненных новостях из test, что и в предыдущем сравнении. Истинные метки известны: исходные абзацы имеют метку `relevant`, вставленные `irrelevant`. Сравним ответы модели с ними.

Задача решается как бинарная классификация, где положительным считается класс `irrelevant`, то есть посторонний абзац. Поэтому метрики имеют следующий смысл:

- **Precision**: какая доля абзацев, которые фильтр счел посторонними, действительно посторонние. Низкая precision означает, что фильтр удаляет полезный текст.
- **Recall**: какая доля посторонних абзацев фильтр нашел. Низкий recall означает, что мусор остается в тексте.
- **F1**: среднее гармоническое precision и recall.
- **Accuracy**: доля абзацев с правильной меткой. Метрика дает завышенную оценку, потому что исходных абзацев больше, чем вставленных.
- **`invalid_json_rate`**: доля ответов, которые не удалось разобрать.


In [ ]:
assert noise_test, 'В test нет загрязненных новостей для сравнения'
test_v3 = []
for record in noise_test[:]:
    item = dict(record['inputs'])
    item['reference'] = record['expectations']['expected_response']
    test_v3.append(item)

baseline_filter = [filter_news(item) for item in test_v3]
trivial = [{'true': row['true'], 'pred': ['relevant'] * len(row['true']), 'invalid': False} for row in baseline_filter]
filter_table = pd.DataFrame(
    {'все relevant': filter_metrics(trivial), 'исходный фильтр': filter_metrics(baseline_filter)}).T
filter_table.loc['все relevant', 'invalid_json_rate'] = np.nan
print(filter_table)

Сравните precision и recall фильтра с правилом «все relevant». Высокий recall означает, что найдено много вставок, но при низкой precision вместе с ними удаляется много полезных абзацев. `invalid_json_rate` показывает отдельно, как часто фильтр не вернул корректный формат. Конкретные значения зависят от модели и текущего запуска.

**Задание 3.** Сравните качество суммаризации без фильтра и с фильтром.

1. Для загрязненных новостей из `test_v3` постройте резюме в трех вариантах: по чистому тексту, по тексту с посторонними абзацами и по тексту после фильтрации.
2. Вычислите ROUGE-1, ROUGE-2 и среднюю длину резюме для каждого варианта и сведите результаты в таблицу.
3. Для каждого из вариантов определите, какую долю удаленных абзацев составляют полезные, то есть сколько исходного текста фильтр потерял.
4. Постройте столбчатую диаграмму ROUGE-1 и ROUGE-2: по горизонтали метрики, столбцы соответствуют трем вариантам.
5. Найдите по одному примеру, когда фильтр удалил полезный абзац и когда оставил посторонний. Как это повлияло на резюме?
6. Сопоставьте качество фильтрации (precision, recall) и качество резюме. Достаточно ли высокого recall, чтобы резюме вернулось к уровню чистых новостей? Объясните, почему.